# 02 · Physical-state detection
Compares A41-A44 for detecting **0 Off, 1 Panel, 2 Started, 3 Cutting** from current for one machine (`MACHINE`), scores them with G1-G6, then builds hours per state and T_op. Related to H45-H47, R36/R37/R42. Run once with `MACHINE = 4`, then again with `MACHINE = 3`; models are always fitted per machine because current levels differ (H45). CSV outputs use Indonesian labels for the report.

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, precision_score, recall_score, cohen_kappa_score
import config as C
from src import detection as D

MACHINE = 3                                  # 4, then 3
SPLIT = pd.Timestamp("2026-08-01")           # before SPLIT: training period (RF, cross-time test)
SAMPLE_DAYS = ["2026-09-08", "2026-10-14"]   # local date

ID = {"A41 Threshold": "A41 Ambang", "accuracy": "akurasi", "precision": "presisi", "flickers_per_day": "kedipan_per_hari",
      "switches_per_day": "perpindahan_per_hari", "cutting_hours": "jam_memotong", "seconds_per_day": "detik_per_hari",
      "score": "skor", "agreement_pct": "kesesuaian_pct", "hold": "tunda", "month": "bulan", "state": "kondisi",
      "G1 label-free": "G1 tanpa label", "G2 accuracy": "G2 akurasi", "G3 manual params": "G3 param manual",
      "G4 s/day": "G4 detik/hari", "G5 flickers/day": "G5 kedipan/hari", "G6 explainability": "G6 keterjelasan",
      "T_op (started+cutting)": "T_op (start+potong)", "T_op (cutting)": "T_op (potong)",
      **{D.STATE[k]: D.LABEL_ID[k] for k in D.STATE}}
save = lambda t, name, **kw: t.rename(index=ID, columns=ID).rename_axis(index=ID.get(t.index.name, t.index.name) or "metode") \
                              .to_csv(C.TAB / f"m{MACHINE}_{name}.csv", **kw)

df = pd.read_parquet(C.SILVER / "timeline.parquet", filters=[("machine", "==", MACHINE)]).reset_index(drop=True)
df["missing"] = ~df.has_data | df.I.isna()
df["day"] = df.t_local.dt.strftime("%Y-%m-%d")
df["month"] = df.t_local.dt.strftime("%Y-%m")
df["excluded"] = df.day.isin(C.TEST_DAYS + C.ANOMALY_DAYS)
valid = (~df.missing & ~df.meter_off).to_numpy()
x = D.log_current(df.I[valid])
n_days = df.day[~df.missing].nunique()
print(f"{len(df):,} slots, {n_days} days with data, {valid.mean():.1%} valid slots (data present, meter on)")

## 1 · Log current and histogram valleys
$x = \log_{10}(I + 0.01)$: levels 0 / 0.1-1 / 2-6 / ~100 A become almost evenly spaced and each is close to normal (basis of GMM/HMM). **4 peaks** should appear.

In [ ]:
valleys, c, hs, peaks = D.find_valleys(x)
print("Peaks (A):", D.to_amps(peaks).round(3), "| Valleys (A):", D.to_amps(valleys).round(3))
if len(peaks) < 4: print("WARNING: fewer than 4 peaks, check the histogram")
amps = [0, 0.1, 0.3, 1, 3, 10, 30, 100, 300]
plt.figure(figsize=(9, 3)); plt.plot(c, hs)
for v in valleys: plt.axvline(v, ls="--", c="r")
plt.xticks(D.log_current(amps), amps)
plt.xlabel("Current I (A), log scale"); plt.ylabel("log(1 + count), smoothed")
plt.title(f"Machine {MACHINE}: how often each current level occurs (red = valleys between states)"); plt.tight_layout()
plt.savefig(C.FIG / f"m{MACHINE}_histogram.png"); plt.show()

## 2 · Four methods
- **A41 Threshold:** up when $x > L_k + \delta$, down when $x < L_{k-1} - \delta$, must hold for `hold` samples (Schmitt trigger + debounce).
- **A42 GMM:** $p(x) = \sum_j \pi_j \mathcal N(x \mid \mu_j, \sigma_j^2)$, $k = 4$ physical states; BIC $= -2\ln\hat L + p\ln n$ as a check.
- **A43 HMM:** GMM + dynamics $P(z_t = j \mid z_{t-1} = i) = a_{ij}$; Viterbi acts like an observer (Ogata) that rejects short spikes. Basis of H47.
- **A44 RF:** supervised with RUNNING labels (trained before 1 Aug), binary only.

In [ ]:
T, S = {}, {}
t0 = time.perf_counter(); k_thr = D.threshold(x, valleys); T["A41 Threshold"] = time.perf_counter() - t0

bic = {k: GaussianMixture(k, random_state=0).fit(x[::7, None]).bic(x[::7, None]) for k in range(2, 8)}
t0 = time.perf_counter(); k_gmm, g = D.gmm(x); T["A42 GMM"] = time.perf_counter() - t0

lengths = D.segments(valid)
t0 = time.perf_counter(); k_hmm, h = D.hmm(x, lengths, g); T["A43 HMM"] = time.perf_counter() - t0

sub = df[valid]
F = sub[["I", "va", "vx", "vy", "vz", "freq"]].fillna(0).to_numpy()
y = (sub.status == "RUNNING").to_numpy()
train = ((sub.t < SPLIT) & sub.status.notna() & ~sub.excluded).to_numpy()
t0 = time.perf_counter()
rf = RandomForestClassifier(100, min_samples_leaf=5, max_samples=min(200_000, train.sum()), n_jobs=-1, random_state=0)
k_rf = np.where(rf.fit(F[train], y[train]).predict(F), 3, 0); T["A44 RF"] = time.perf_counter() - t0

def to_grid(k):
    s = np.full(len(df), -1, np.int8); s[df.meter_off.to_numpy()] = 0; s[valid] = k; return s
S = {m: to_grid(k) for m, k in zip(T, [k_thr, k_gmm, k_hmm, k_rf])}
print("Lowest BIC at k =", min(bic, key=bic.get))
print("GMM state means (A):", D.to_amps(np.sort(g.means_.ravel())).round(3))
print(f"HMM converged: {h.monitor_.converged} ({h.monitor_.iter} iterations)")
print({m: f"{t:.1f} s" for m, t in T.items()})

## 3 · Evaluation
Test set: from 1 Aug, status present, test/anomaly days excluded. *Cutting* is compared with *RUNNING*. RUNNING itself is imperfect (P30), so this is **agreement with the label**, not absolute truth.
**Flicker** = Cutting run ≤ 10 s; each one can become a false start-stop event that lowers MTBF.

In [ ]:
test = ((df.t >= SPLIT) & ~df.missing & df.status.notna() & ~df.excluded).to_numpy()
yt = (df.status == "RUNNING").to_numpy()
ev = pd.DataFrame({m: {
        "accuracy": accuracy_score(yt[test], s[test] == 3), "precision": precision_score(yt[test], s[test] == 3),
        "recall": recall_score(yt[test], s[test] == 3), "kappa": cohen_kappa_score(yt[test], s[test] == 3),
        "flickers_per_day": D.flickers(s) / n_days, "switches_per_day": np.sum(s[1:] != s[:-1]) / n_days,
        "cutting_hours": np.sum(s == 3) * 5 / 3600, "seconds_per_day": T[m] / n_days}
      for m, s in S.items()}).T
display(ev.round(4))

ok = ~df.missing & df.status.notna() & ~df.excluded
monthly = pd.DataFrame({m: pd.Series((s == 3) == yt)[ok].groupby(df.month[ok]).mean() for m, s in S.items()})
display((monthly * 100).round(2))   # RF in Jun-Jul is scored on its own training data (optimistic)
save(ev, "evaluasi"); save(monthly, "kecocokan_bulanan")

## 4 · G1-G6 score
$s_i = \mathrm{clip}\big((v_i - v_{und})/(v_{des} - v_{und}), 0, 1\big)$, score $= \sum w_i s_i$, with the same limits and weights as the Proyek sheet. Then 10,000 random weight sets test whether the winner depends on the weights.

In [ ]:
K = pd.DataFrame({"G1 label-free": (0, 1, .15), "G2 accuracy": (.971, 1, .35), "G3 manual params": (3, 0, .10),
                  "G4 s/day": (5, 0, .05), "G5 flickers/day": (20, 0, .20), "G6 explainability": (0, 1, .15)},
                 index=["und", "des", "w"]).T
V = pd.DataFrame({"G1 label-free": [1, 1, 1, 0], "G2 accuracy": ev.accuracy, "G3 manual params": [3, 0, 0, 0],
                  "G4 s/day": ev.seconds_per_day, "G5 flickers/day": ev.flickers_per_day,
                  "G6 explainability": [1, .5, .5, 0]}, index=ev.index)
N = ((V - K.und) / (K.des - K.und)).clip(0, 1)
score = (N @ K.w).sort_values(ascending=False).rename("score")
wins = pd.Series([(N @ w).idxmax() for w in np.random.default_rng(0).dirichlet(np.ones(6), 10_000)]).value_counts(normalize=True)
display(pd.concat([V, score, (wins * 100).rename("pct_wins")], axis=1).sort_values("score", ascending=False).round(3))
save(pd.concat([V, N.rename(columns=ID).add_suffix(" (norm)"), score], axis=1), "skor")

## 5 · Robustness tests
- **Cross-time:** models fitted only on data before 1 Aug are applied after it and compared with full-data models. High agreement means the model can be reused for later patches.
- **A41 sensitivity:** δ and hold are varied.

In [ ]:
early = (df.t[valid] < SPLIT).to_numpy(); late = ~early
len_early = D.segments(valid & (df.t < SPLIT).to_numpy())
k_thr2 = D.threshold(x, D.find_valleys(x[early])[0])
k_gmm2, g2 = D.gmm(x, x_fit=x[early])
k_hmm2, _ = D.hmm(x, lengths, g2, x_fit=x[early], len_fit=len_early)
cross = pd.Series({"A41 Threshold": np.mean(k_thr2[late] == k_thr[late]), "A42 GMM": np.mean(k_gmm2[late] == k_gmm[late]),
                   "A43 HMM": np.mean(k_hmm2[late] == k_hmm[late])}, name="agreement_pct") * 100
display(cross.round(3))

sens = pd.DataFrame([{"delta": d, "hold": n, "accuracy": accuracy_score(yt[test], to_grid(kk)[test] == 3),
                      "flickers_per_day": D.flickers(to_grid(kk)) / n_days}
                     for d in (.02, .05, .10) for n in (1, 2, 3, 6) for kk in [D.threshold(x, valleys, d, n)]])
display(sens.round(4))
save(cross.to_frame(), "lintas_waktu"); save(sens, "sensitivitas_ambang", index=False)

## 6 · Reconstruction
The best-scoring label-free method is used for now; the final choice comes after machine 3. T_op is given two ways for Q137/R42: **Started + Cutting** and **Cutting only**.

In [ ]:
CHOSEN = score.drop("A44 RF").idxmax()          # override if needed, e.g. CHOSEN = "A43 HMM"
s = S[CHOSEN]; state = pd.Series(s).map(D.STATE).rename("state")
hours = pd.crosstab(df.month, state) * 5 / 3600
hours["T_op (started+cutting)"] = hours.get("Started", 0) + hours.get("Cutting", 0)
hours["T_op (cutting)"] = hours.get("Cutting", 0)
hours.loc["Total"] = hours.sum()
print("Method:", CHOSEN); display(hours.round(1))

m = ~df.missing & df.status.notna()
cross_status = pd.crosstab(df.status[m], state[m]) * 5 / 3600
display(cross_status.round(1)); display((cross_status.div(cross_status.sum(axis=1), axis=0) * 100).round(1))
save(hours, "jam_kondisi"); save(cross_status, "kondisi_vs_status")

Visual check: do state changes line up with current changes, and where do the methods disagree?

In [ ]:
for day in SAMPLE_DAYS:
    m = (df.day == day).to_numpy()
    if not m.any(): print("no data for", day); continue
    fig, ax = plt.subplots(4, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1, 1, 1]})
    ax[0].semilogy(df.t_local[m], df.I[m] + D.EPS, lw=.6); ax[0].set_ylabel("Current (A)")
    ax[0].set_title(f"Machine {MACHINE}, {day}: current (top) and detected state per method")
    for a, k in zip(ax[1:], ["A41 Threshold", "A42 GMM", "A43 HMM"]):
        a.step(df.t_local[m], S[k][m], where="post", lw=.8); a.set_ylabel(k.split()[1])
        a.set_yticks(range(-1, 4), ["No data", "Off", "Panel", "Started", "Cutting"], fontsize=7)
    ax[-1].set_xlabel("Local time (WIB)")
    fig.tight_layout(); fig.savefig(C.FIG / f"m{MACHINE}_visual_{day}.png"); plt.show()

## 7 · Save

In [ ]:
out = df[["machine", "t"]].assign(**{f"k_{k.split()[1].lower()}": v for k, v in S.items()})
out.to_parquet(C.SILVER / f"states_m{MACHINE}.parquet", index=False)
v, d, i = D.runs(s)
ep = pd.DataFrame({"start": df.t.to_numpy()[i], "state": [D.STATE[int(k)] for k in v], "duration_s": d * 5})
ep.to_parquet(C.SILVER / f"episodes_m{MACHINE}.parquet", index=False)
print(f"states_m{MACHINE}.parquet ({len(out):,} slots), episodes_m{MACHINE}.parquet ({len(ep):,} episodes, {CHOSEN})")
print(f"H47: flickers/day GMM {ev.loc['A42 GMM','flickers_per_day']:.2f} vs HMM {ev.loc['A43 HMM','flickers_per_day']:.2f}; "
      f"switches/day GMM {ev.loc['A42 GMM','switches_per_day']:.1f} vs HMM {ev.loc['A43 HMM','switches_per_day']:.1f}")